In [ ]:
!pip install admet_ai


In [ ]:
import torch
import numpy as np
from argparse import Namespace
from admet_ai import ADMETModel
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, Crippen

torch.serialization.add_safe_globals([
    Namespace,
    np.core.multiarray._reconstruct,
    np.ndarray,
    np.dtype,
    np.dtypes.Float64DType
])

binding_scores = [0.4914195347628156, 0.8303721053466643, 1.0581474159307722, 0.7795295945044215, 0.3308276468591592, 0.9519032299358294, 1.1030335677176402, 0.7674287004159897]
ligand_smiles_list = ['CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(=N3)C)N4CCN(CC4)CCO', 'CC1=C(C=C(C=C1)C(=O)NC2=CC(=C(C=C2)CN3CCN(CC3)C)C(F)(F)F)C#CC4=CN=C5N4N=CC=C5', 'C1CN(C[C@@H]1O)C2=C(C=C(C=N2)C(=O)NC3=CC=C(C=C3)OC(F)(F)Cl)C4=CC=NN4', 'CNC(=O)C1=CC=CC=C1SC2=CC3=C(C=C2)C(=NN3)/C=C/C4=CC=CC=N4', 'CC1=C(C=C(C=C1)C(=O)NC2=CC(=CC(=C2)C(F)(F)F)N3C=C(N=C3)C)NC4=NC=CC(=N4)C5=CN=CC=C5', 'CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5', 'CC1=C(C=C(C=C1)NC(=O)C2=CC(=C(C=C2)CN3CC[C@@H](C3)N(C)C)C(F)(F)F)NC4=NC=CC(=N4)C5=CN=CN=C5', 'CN1CCN(CC1)CCCOC2=C(C=C3C(=C2)N=CC(=C3NC4=CC(=C(C=C4Cl)Cl)OC)C#N)OC']

model = ADMETModel()
admet_df = model.predict(smiles=ligand_smiles_list)

admet_df = admet_df.reset_index().rename(columns={'index': 'SMILES'})

def lipinski_ok(mol):
    return (Descriptors.MolWt(mol) < 500 and
            Crippen.MolLogP(mol) < 5 and
            Lipinski.NumHDonors(mol) <= 5 and
            Lipinski.NumHAcceptors(mol) <= 10)

lipinski_flags = [lipinski_ok(Chem.MolFromSmiles(smi)) for smi in admet_df['SMILES']]
admet_df["lipinski_pass"] = lipinski_flags

good_admet = (
    (admet_df["HIA_Hou"] == True) &
    (admet_df["BBB_Martins"] == False) &
    (admet_df["hERG"] == False) &
    (admet_df["CYP3A4_Veith"] == False) &
    (admet_df["lipinski_pass"])
)
admet_df["admet_pass"] = good_admet

# --- Combine with binding scores and sort ---
rank_df = (
    admet_df
    .assign(binding_score=binding_scores)
    .sort_values(["admet_pass", "binding_score"], ascending=[False, False])
    .reset_index(drop=True)
)

print("--- DETAILED ADMET PROFILE ---")
print(rank_df[[
    "SMILES",
    "binding_score",
    "HIA_Hou",
    "BBB_Martins",
    "hERG",
    "CYP3A4_Veith",
    "lipinski_pass",
    "admet_pass" # The final combined result
]].head(10))